## **正規表達式**半導體**日誌解析器**

- 目標：精通 Python 正規表達式（Regex）的「命名捕獲群組（Named Capture Groups）」語法，編寫高效的 LOG_PATTERN 來解析含有時間戳（Timestamp）、日誌層級（Level）、模組名稱（Module）與核心訊息（Message）的半導體設備日誌。


### 1. 半導體常見日誌格式分析

- 核心：
    - 在半導體自動化測試或機台通訊（如 SECS/GEM、VISA/SCPI）中，文字日誌通常包含以下固定結構。為了能用 Pandas 進行巨量分析，必須將其解析成**表格**：
        - 時間戳 (Timestamp)：例如 2026-08-04 14:22:05,123
        - 日誌層級 (Level)：例如 [INFO], [WARN], [ERROR]
        - 來源模組 (Module)：例如 [ProbeCardController], [YieldPredictor]
        - 核心訊息 (Message)：例如 Connection established to IP 192.168.1.100


### 2. 正規表達式編寫與 parser.py 核心邏輯

- 實作：我們使用 Python 的 **re 模組**，並透過 (?P<name>pattern) 語法為每個**欄位命名**，這能大幅提高程式碼的可讀性與維護性。


In [ ]:
import re
import pandas as pd
from datetime import datetime

# --- 定義半導體測試機台的日誌模式 (LOG_PATTERN) ---
# 語法拆解：
# (?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\,\d{3}) -> 匹配 2026-08-04 14:22:05,123
# \s+\[(?P<level>\w+)\] -> 匹配 空格 + [INFO]
# \s+\[(?P<module>\w+)\] -> 匹配 空格 + [ProbeCardController]
# \s+-\s+(?P<message>.*) -> 匹配 破折號後面的所有訊息
LOG_PATTERN = re.compile(
    r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\,\d{3})"
    r"\s+\[(?P<level>\w+)\]"
    r"\s+\[(?P<module>\w+)\]"
    r"\s+-\s+(?P<message>.*)"
)

# 模擬從測試機台或資料夾中讀取到的原始髒日誌陣列
raw_logs = [
    "2026-08-04 14:22:05,123 [INFO] [ProbeCardController] - Connection established to IP 192.168.1.100",
    "2026-08-04 14:22:06,456 [WARN] [HardwareInterface] - Voltage drop detected: 1.15V below threshold",
    "2026-08-04 14:22:08,789 [ERROR] [YieldPredictor] - WaferBinMap file 'LOT_A_W05.json' missing",
    "髒資料行：這行是機台當機噴出的雜訊，不符合格式",
    "2026-08-04 14:22:10,001 [INFO] [ProbeCardController] - Retry connection count: 1",
]


def parse_semiconductor_log(logs_list):
    """解析日誌清單並回傳結構化資料"""
    parsed_records = []

    for line in logs_list:
        match = LOG_PATTERN.match(line)
        if match:
            # 透過 .groupdict() 直接將捕獲的欄位轉成 Python 字典
            data = match.groupdict()
            # 將時間戳字串轉換為 datetime 物件，方便後續時間序列分析
            data["timestamp"] = datetime.strptime(
                data["timestamp"], "%Y-%m-%d %H:%M:%S,%True"
            )
            parsed_records.append(data)
        else:
            # 記錄不符合標準格式的髒資料，面試時的大加分亮點
            print(f"[Parser Warning] 發現無法解析的髒日誌行: '{line}'")

    return parsed_records


# 執行解析
print(">>> 開始執行日誌解析 Pipeline...")
structured_data = parse_semiconductor_log(raw_logs)

### 4. 將解析結果轉換為 Pandas DataFrame 進行過濾

- 核心：解析完成後，直接將字典清單丟入 Pandas，就能輕鬆進行過濾與統計。


In [ ]:
# 建立 DataFrame
df_log = pd.DataFrame(structured_data)

print("\n 成功轉化為結構化 DataFrame:")
print(df_log)

# 找出所有 [ERROR] 或 [WARN] 的嚴重機台事件
print("\n 篩選出需要觸發 Alert 的異常事件：")
critical_events = df_log[df_log["level"].isin(["ERROR", "WARN"])]
print(critical_events[["timestamp", "module", "message"]])

# 統計每個模組發生的事件次數
print("\n 各模組事件次數統計：")
print(df_log["module"].value_counts())

- 總結：在我的專案中，機台產生的原始日誌通常是無結構的文字。為了打造自動化分析 Pipeline，我編寫了 parser.py。我摒棄了簡單的 split() 字串切割，因為訊息欄位可能包含空格或破折號會導致切割錯位；我採用 Regex 的命名捕獲群組（Named Capture Groups）。這不僅讓 LOG_PATTERN 极具可讀性，還能透過 .groupdict() 快速將資料導成字典，一秒轉換為 Pandas DataFrame。同時，我的 Parser 設計了容錯機制，能自動過濾不合規的雜訊行並將其拋出警告，確保後續機器學習模型的輸入資料絕對乾淨。
